# Purchasing Cycle Comparison

The inventory replenishment team wants to compare the typical purchasing cycles of Produce and Alcohol shoppers.

For this analysis, each user's purchasing cycle is defined as the median `days_since_prior_order` across their qualifying department orders. The median of these user-level cycles is then used to represent the typical shopper in each department.

This approach gives each shopper equal weight and prevents customers with many orders from dominating the comparison.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


DATA_DIR = "../../data/shop"


orders = pd.read_csv(
    f"{DATA_DIR}/orders.csv",
    usecols=[
        "order_id",
        "user_id",
        "days_since_prior_order",
    ],
)

order_products = pd.read_csv(
    f"{DATA_DIR}/order_products__prior.csv",
    usecols=[
        "order_id",
        "product_id",
    ],
)

products = pd.read_csv(
    f"{DATA_DIR}/products.csv",
    usecols=[
        "product_id",
        "department_id",
    ],
)

departments = pd.read_csv(
    f"{DATA_DIR}/departments.csv",
    usecols=[
        "department_id",
        "department",
    ],
)


# Identify Produce and Alcohol orders.
department_orders = (
    order_products
    .merge(
        products,
        on="product_id",
    )
    .merge(
        departments,
        on="department_id",
    )
    .query(
        "department in ['produce', 'alcohol']"
    )
    [
        [
            "order_id",
            "department",
        ]
    ]
    .drop_duplicates()
    .merge(
        orders,
        on="order_id",
    )
    .dropna(
        subset=["days_since_prior_order"]
    )
)

# One typical cycle per user.
user_cycles = (
    department_orders
    .groupby(
        [
            "department",
            "user_id",
        ],
        as_index=False,
    )
    .agg(
        user_cycle_days=(
            "days_since_prior_order",
            "median",
        ),
    )
)


# Typical customer-level cycle.
correct_summary = (
    user_cycles
    .groupby(
        "department",
        as_index=False,
    )
    .agg(
        user_count=(
            "user_id",
            "nunique",
        ),
        typical_cycle_days=(
            "user_cycle_days",
            "median",
        ),
    )
)


print(correct_summary.round(2))


produce_cycle = correct_summary.loc[
    correct_summary["department"].eq("produce"),
    "typical_cycle_days",
].iloc[0]

alcohol_cycle = correct_summary.loc[
    correct_summary["department"].eq("alcohol"),
    "typical_cycle_days",
].iloc[0]


shorter_group = (
    "Produce"
    if produce_cycle < alcohol_cycle
    else "Alcohol"
)

difference = abs(
    produce_cycle - alcohol_cycle
)


print(
    f"\nShorter purchasing cycle: {shorter_group}"
)

print(
    f"Difference: {difference:.2f} days"
)

In [ ]:
plot_data = correct_summary.copy()

plot_data["department"] = (
    plot_data["department"]
    .str.title()
)


fig, ax = plt.subplots(
    figsize=(7, 5)
)

ax.bar(
    plot_data["department"],
    plot_data["typical_cycle_days"],
)

ax.set_title(
    "Typical Customer Purchasing Cycle"
)

ax.set_ylabel(
    "Median User-level Cycle (Days)"
)


for index, value in enumerate(
    plot_data["typical_cycle_days"]
):
    ax.text(
        index,
        value,
        f"{value:.2f}",
        ha="center",
        va="bottom",
    )


plt.tight_layout()
plt.show()

  department  user_count  typical_cycle_days
0    alcohol       14479                10.5
1    produce      190803                12.0

Shorter purchasing cycle: Alcohol
Difference: 1.50 days


## Conclusion

Using the customer-level definition, Alcohol shoppers have the shorter typical purchasing cycle.

The median user-level cycle is **10.5 days** for Alcohol shoppers and **12.0 days** for Produce shoppers, a difference of **1.5 days**.

This suggests that Alcohol shoppers return slightly sooner on average under the selected customer-level definition.
